# Magnetic Field Evolution in Satellite Galaxies
## PHY 225-001 — Computational Physics Project

**Goal:** Track magnetic field strength as a function of redshift for satellite galaxies in TNG100, using merger tree data to identify the moment of central-to-satellite transition (infall time).

---
### Notebook Structure
1. Setup & API Configuration
2. Query Satellite Galaxies at z=0
3. Walk the Merger Tree (main progenitor branch)
4. Extract Magnetic Field Strength at Each Snapshot
5. Identify Infall Time (central → satellite transition)
6. Plot B-field vs Redshift (single galaxy)
7. Generalize to Multiple Galaxies
8. Averaged B-field vs Time Since Infall

---
## 1. Setup & API Configuration

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import requests
import astropy.units as u
from astropy.cosmology import FlatLambdaCDM
from scipy.interpolate import interp1d
from scipy.ndimage import uniform_filter1d

# ── TNG100 cosmological parameters (Planck 2015) ──────────────────────────────
cosmo = FlatLambdaCDM(H0=67.74, Om0=0.3089)

# ── API configuration ─────────────────────────────────────────────────────────
# Paste your TNG API key here (register at: https://www.tng-project.org/users/register/)
API_KEY  = "f47c8519062d199613c99b12244d62ab"  
BASE_URL = "https://www.tng-project.org/api/"
SIM_NAME = "TNG100-1"            # highest-resolution TNG100 run

HEADERS  = {"api-key": API_KEY}

def get(path, params=None):
    """Thin wrapper around requests.get that handles TNG API auth and errors."""
    url = path if path.startswith("http") else BASE_URL + path
    r = requests.get(url, params=params, headers=HEADERS)
    r.raise_for_status()
    return r.json()

# Quick connectivity test
try:
    info = get(f"{BASE_URL}{SIM_NAME}/")
    print(f"Connected to {SIM_NAME}")
    print(f"  Number of snapshots : {info['num_snapshots']}")
    print(f"  Box size            : {info['box_size']:.1f} ckpc/h")
except Exception as e:
    print(f"Connection failed — check your API key: {e}")

Connection failed — check your API key: HTTPSConnectionPool(host='www.tng-project.org', port=443): Max retries exceeded with url: /api/TNG100-1/ (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x77fdb278d160>, 'Connection to www.tng-project.org timed out. (connect timeout=None)'))


---
## 2. Build the Snapshot → Redshift / Lookback-Time Table

TNG100 has **100 snapshots** (snap 0 = high-z, snap 99 = z=0).  
We download the metadata once and store it for all later conversions.

In [ ]:
def build_snapshot_table(sim_name=SIM_NAME):
    """Return a dict mapping snapshot number → {redshift, scale_factor, lookback_time_Gyr}."""
    snaps_info = get(f"{BASE_URL}{sim_name}/snapshots/")
    table = {}
    for s in snaps_info:
        z  = s["redshift"]
        lb = cosmo.lookback_time(z).to(u.Gyr).value
        table[s["number"]] = {
            "redshift"        : z,
            "scale_factor"    : 1.0 / (1.0 + z),
            "lookback_time"   : lb,        # Gyr
        }
    return table

SNAP_TABLE = build_snapshot_table()

# Preview a few rows
print(f"{'Snap':>5}  {'z':>8}  {'a':>8}  {'Lookback (Gyr)':>15}")
print("-" * 45)
for sn in [0, 10, 33, 50, 67, 78, 91, 99]:
    row = SNAP_TABLE[sn]
    print(f"{sn:>5}  {row['redshift']:>8.3f}  {row['scale_factor']:>8.4f}  {row['lookback_time']:>15.3f}")

---
## 3. Query Satellite Galaxies at z = 0 (Snapshot 99)

We select **satellite** subhalos from the z=0 snapshot.  
In TNG, a subhalo is a **satellite** if it is NOT the most-massive subhalo in its Friends-of-Friends (FoF) halo — i.e., `SubhaloGrNr` is set but the subhalo is not the primary (`SubhaloRankInGrLen > 0`).  

The API exposes this via the `primary_flag` field: `primary_flag=0` → satellite.

In [ ]:
def query_satellites(snap=99, stellar_mass_min_log=9.0, stellar_mass_max_log=11.5,
                     limit=20):
    """
    Return a list of satellite subhalo dicts from a given snapshot.

    Parameters
    ----------
    snap              : TNG snapshot number (99 = z=0)
    stellar_mass_min_log : log10(M*/Msun) lower bound
    stellar_mass_max_log : log10(M*/Msun) upper bound
    limit             : max number of subhalos to return (start small!)
    """
    # IllustrisTNG stores masses in units of 1e10 Msun/h  (h=0.6774)
    h = 0.6774
    mass_unit = 1e10 / h   # Msun per code unit

    m_min = 10**stellar_mass_min_log / mass_unit
    m_max = 10**stellar_mass_max_log / mass_unit

    params = {
        "primary_flag"          : 0,          # satellites only
        "SubhaloMassType__gte"  : f"4,{m_min:.6f}",   # stellar mass > min (type 4 = stars)
        "SubhaloMassType__lte"  : f"4,{m_max:.6f}",
        "fields"                : ("id,SubhaloMassType,SubhaloMagneticField,"
                                   "SubhaloGrNr,SubhaloHalfmassRadType"),
        "limit"                 : limit,
    }

    result = get(f"{BASE_URL}{SIM_NAME}/snapshots/{snap}/subhalos/", params=params)
    galaxies = result.get("results", [])
    print(f"Found {result['count']} satellites matching criteria; returning {len(galaxies)}.")
    return galaxies

# Start with a small sample while you explore
satellites_z0 = query_satellites(snap=99, stellar_mass_min_log=9.5, limit=10)

# Show basic info
h = 0.6774
for g in satellites_z0[:5]:
    M_star = g["SubhaloMassType"][4] * 1e10 / h   # convert to Msun
    B_gauss = g["SubhaloMagneticField"]            # comoving Gauss in TNG
    print(f"  SubhaloID={g['id']:>8d}  log M*={np.log10(M_star):.2f}  B={B_gauss:.4e} G")

---
## 4. Walk the Merger Tree — Main Progenitor Branch

The **Main Progenitor Branch (MPB)** traces the most massive progenitor at every snapshot.  
The TNG API provides this directly via the `/sublink/mpb` endpoint — no manual tree traversal needed.

**Fields we extract at each snapshot:**
| Field | Description |
|---|---|
| `SnapNum` | Snapshot number |
| `SubhaloMagneticField` | Volume-weighted mean B-field (comoving Gauss) |
| `SubhaloMassType` | Mass by particle type (index 4 = stellar) |
| `GroupFirstSub` | SubhaloID of the primary subhalo of the host FoF group |
| `SubfindID` | The subhalo's own ID at that snapshot |

> **Note on units:** `SubhaloMagneticField` is in comoving Gauss. To convert to physical Gauss: B_phys = B_com / a² (where a is the scale factor). TNG actually outputs the value already divided by a² in some versions — confirm by checking the [TNG fields documentation](https://www.tng-project.org/data/docs/specifications/).

In [ ]:
def get_mpb(subhalo_id, snap=99):
    """
    Retrieve the Main Progenitor Branch for a subhalo.

    Returns a list of dicts, one per snapshot (ordered from z=0 back in time),
    each containing the fields we need for analysis.
    """
    fields = "SnapNum,SubhaloMagneticField,SubhaloMassType,SubfindID,GroupFirstSub"
    url    = f"{BASE_URL}{SIM_NAME}/snapshots/{snap}/subhalos/{subhalo_id}/sublink/mpb/"
    result = get(url, params={"fields": fields})

    records = []
    n = len(result["SnapNum"])
    for i in range(n):
        sn        = result["SnapNum"][i]
        snap_data = SNAP_TABLE.get(sn, {})
        records.append({
            "snap"          : sn,
            "redshift"      : snap_data.get("redshift", np.nan),
            "lookback_time" : snap_data.get("lookback_time", np.nan),
            "scale_factor"  : snap_data.get("scale_factor", np.nan),
            "B_comoving"    : result["SubhaloMagneticField"][i],
            "M_star"        : result["SubhaloMassType"][i][4] * 1e10 / 0.6774,  # Msun
            "subfind_id"    : result["SubfindID"][i],
            "group_first_sub": result["GroupFirstSub"][i],
        })
    return records


def compute_B_physical(records):
    """
    Add B_physical (Gauss) to each record.
    B_phys = B_comoving / a^2
    (Check TNG documentation for your specific field definition.)
    """
    for r in records:
        a = r["scale_factor"]
        r["B_physical"] = r["B_comoving"] / a**2 if a > 0 else np.nan
    return records


# Test on the first satellite
test_id  = satellites_z0[0]["id"]
mpb_data = get_mpb(test_id)
mpb_data = compute_B_physical(mpb_data)

print(f"MPB for SubhaloID {test_id}: {len(mpb_data)} snapshots")
print(f"{'Snap':>5}  {'z':>7}  {'B_com (G)':>12}  {'B_phys (G)':>12}  {'log M*':>8}")
print("-" * 55)
for r in mpb_data[:8]:
    print(f"{r['snap']:>5}  {r['redshift']:>7.3f}  "
          f"{r['B_comoving']:>12.4e}  {r['B_physical']:>12.4e}  "
          f"{np.log10(r['M_star'] + 1e-5):>8.2f}")

---
## 5. Identify Infall Time (Central → Satellite Transition)

A galaxy **becomes a satellite** at the snapshot when it stops being the primary subhalo of its FoF group.  
In practice: we walk the MPB from z=0 **backwards** and find the first snapshot where `subfind_id == group_first_sub` (i.e., it was still a central).  
The **infall snapshot** is the one just before that.

In [ ]:
def find_infall_snap(mpb_records):
    """
    Identify the snapshot of central→satellite transition.

    Walk the MPB from z=0 (index 0) toward higher redshift.
    Return the snapshot number and lookback time at infall,
    or None if the galaxy was always a satellite (or always a central).
    """
    # MPB is ordered z=0 → high-z (index 0 is z=0)
    was_central_at = None
    for i, r in enumerate(mpb_records):
        is_central = (r["subfind_id"] == r["group_first_sub"])
        if is_central:
            was_central_at = i   # track last (most-recent) snapshot where it was central

    if was_central_at is None:
        return None   # always a satellite — skip or handle separately

    if was_central_at == 0:
        return None   # still a central at z=0 — shouldn't happen if we filtered satellites

    # Infall happened between snapshot was_central_at and was_central_at-1
    # (going forward in time). We take was_central_at-1 as the infall snap.
    infall_record = mpb_records[was_central_at - 1]
    return {
        "snap"         : infall_record["snap"],
        "redshift"     : infall_record["redshift"],
        "lookback_time": infall_record["lookback_time"],
    }


infall = find_infall_snap(mpb_data)
if infall:
    print(f"Infall snapshot : {infall['snap']}")
    print(f"Infall redshift : z = {infall['redshift']:.3f}")
    print(f"Infall lookback : {infall['lookback_time']:.2f} Gyr ago")
else:
    print("Could not determine infall time for this galaxy.")

---
## 6. Plot B-field vs Redshift — Single Galaxy

We now have everything we need for an initial plot. We'll:
- Plot B_physical vs redshift along the MPB
- Mark the infall redshift with a vertical line
- Optionally overlay an interpolated smooth curve

In [ ]:
def plot_B_vs_redshift_single(mpb_records, infall_info=None, subhalo_id=None,
                              smooth=True):
    """
    Plot magnetic field strength vs redshift for a single galaxy's MPB.

    Parameters
    ----------
    mpb_records  : list of dicts from get_mpb() + compute_B_physical()
    infall_info  : dict from find_infall_snap(), or None
    subhalo_id   : int, used for plot title
    smooth       : if True, also plot a smoothed curve
    """
    # Pull out arrays (filter out NaN or zero B-field)
    records_valid = [r for r in mpb_records
                     if np.isfinite(r["B_physical"]) and r["B_physical"] > 0]

    z_arr = np.array([r["redshift"]   for r in records_valid])
    B_arr = np.array([r["B_physical"] for r in records_valid])

    # Sort by increasing redshift for clean plotting
    order = np.argsort(z_arr)
    z_arr, B_arr = z_arr[order], B_arr[order]

    # ── Interpolation onto a fine grid ───────────────────────────────────────
    z_fine = np.linspace(z_arr.min(), z_arr.max(), 500)
    B_interp = interp1d(z_arr, B_arr, kind="cubic", fill_value="extrapolate")
    B_fine   = B_interp(z_fine)

    # ── Plot ─────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 5))

    ax.scatter(z_arr, B_arr, s=30, color="steelblue", zorder=3,
               label="TNG100 snapshots")
    ax.plot(z_arr, B_arr, color="steelblue", alpha=0.4, lw=1)

    if smooth:
        ax.plot(z_fine, B_fine, color="navy", lw=2, ls="--",
                label="Cubic interpolation")

    # Infall marker
    if infall_info:
        ax.axvline(infall_info["redshift"], color="tomato", lw=2, ls="-",
                   label=f"Infall  z = {infall_info['redshift']:.2f}")

    ax.set_yscale("log")
    ax.set_xlabel("Redshift $z$", fontsize=13)
    ax.set_ylabel(r"$B_{\rm phys}$ [G]", fontsize=13)
    title = f"Magnetic Field Evolution — Subhalo {subhalo_id}" if subhalo_id else "Magnetic Field Evolution"
    ax.set_title(title, fontsize=14)
    ax.legend(fontsize=11)
    ax.invert_xaxis()   # redshift increases to the left (older universe on right)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("results/figures/B_vs_z_single.png", dpi=150)
    plt.show()


# Run the plot
plot_B_vs_redshift_single(mpb_data, infall_info=infall, subhalo_id=test_id)

---
## 7. Generalize to Multiple Galaxies

Now we loop over all queried satellites, fetch each MPB, and store the results.

In [ ]:
def process_all_satellites(satellite_list):
    """
    For each satellite, fetch MPB, compute physical B-field, and find infall.

    Returns a list of dicts, one per galaxy, each containing:
        - subhalo_id
        - mpb       : list of per-snapshot records
        - infall    : infall info dict (or None)
    """
    results = []
    for i, gal in enumerate(satellite_list):
        sid = gal["id"]
        print(f"[{i+1}/{len(satellite_list)}] Processing SubhaloID {sid} ...", end=" ")
        try:
            mpb    = get_mpb(sid)
            mpb    = compute_B_physical(mpb)
            infall = find_infall_snap(mpb)
            results.append({"subhalo_id": sid, "mpb": mpb, "infall": infall})
            print(f"OK  (infall z={infall['redshift']:.2f})" if infall else "OK (no infall found)")
        except Exception as e:
            print(f"FAILED: {e}")
    return results


all_galaxies = process_all_satellites(satellites_z0)
print(f"\nSuccessfully processed: {len(all_galaxies)} galaxies")
print(f"With identified infall: {sum(1 for g in all_galaxies if g['infall'])}")

---
## 8. Multi-Galaxy Overlay — B-field vs Redshift

In [ ]:
def plot_B_vs_z_all(all_galaxies, highlight_infall=True):
    """
    Overlay MPB B-field tracks for all galaxies on one plot.
    Color-code by infall redshift.
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    cmap    = plt.cm.plasma
    infalls = [g["infall"]["redshift"] for g in all_galaxies if g["infall"]]
    z_min, z_max = (min(infalls), max(infalls)) if infalls else (0, 2)
    norm    = plt.Normalize(vmin=z_min, vmax=z_max)

    for gal in all_galaxies:
        mpb    = gal["mpb"]
        infall = gal["infall"]
        valid  = [r for r in mpb if np.isfinite(r["B_physical"]) and r["B_physical"] > 0]
        if not valid:
            continue

        z_arr = np.array([r["redshift"]   for r in valid])
        B_arr = np.array([r["B_physical"] for r in valid])
        order = np.argsort(z_arr)

        color = cmap(norm(infall["redshift"])) if infall else "gray"
        ax.plot(z_arr[order], B_arr[order], color=color, alpha=0.55, lw=1.5)

        if highlight_infall and infall:
            ax.axvline(infall["redshift"], color=color, lw=0.6, ls=":", alpha=0.4)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label="Infall redshift $z_{\\rm infall}$")

    ax.set_yscale("log")
    ax.set_xlabel("Redshift $z$", fontsize=13)
    ax.set_ylabel(r"$B_{\rm phys}$ [G]", fontsize=13)
    ax.set_title("Magnetic Field Evolution — All Sampled Satellites (TNG100)", fontsize=14)
    ax.invert_xaxis()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("results/figures/B_vs_z_all.png", dpi=150)
    plt.show()


plot_B_vs_z_all(all_galaxies)

---
## 9. B-field vs Time Since Infall (Key Science Plot)

This is the central result: re-align all tracks so that **t = 0 corresponds to infall**,
then average across galaxies to reveal the characteristic amplification timescale.

In [ ]:
def compute_dt_infall(mpb_records, infall_info):
    """
    Compute Δt = lookback_time - lookback_time_infall for each snapshot.
    Positive Δt → before infall (higher redshift).
    Negative Δt → after infall (lower redshift, closer to today).
    """
    if infall_info is None:
        return None
    t_inf = infall_info["lookback_time"]
    for r in mpb_records:
        r["dt_infall"] = r["lookback_time"] - t_inf
    return mpb_records


def plot_B_vs_dt_infall(all_galaxies, dt_range=(-4, 6), bin_width=0.5):
    """
    Plot mean B-field as a function of time since infall.

    Parameters
    ----------
    dt_range  : (min, max) Gyr relative to infall (negative = after infall)
    bin_width : width of time bins in Gyr
    """
    # ── Collect all (dt, B) pairs ─────────────────────────────────────────
    all_dt, all_B = [], []
    for gal in all_galaxies:
        if not gal["infall"]:
            continue
        mpb = compute_dt_infall(gal["mpb"], gal["infall"])
        if mpb is None:
            continue
        for r in mpb:
            if np.isfinite(r["B_physical"]) and r["B_physical"] > 0:
                all_dt.append(r["dt_infall"])
                all_B.append(r["B_physical"])

    all_dt = np.array(all_dt)
    all_B  = np.array(all_B)

    # ── Bin and compute median + 16th/84th percentile (1-sigma equivalent) ─
    bins   = np.arange(dt_range[0], dt_range[1] + bin_width, bin_width)
    bin_c  = 0.5 * (bins[:-1] + bins[1:])
    median, p16, p84 = [], [], []

    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (all_dt >= lo) & (all_dt < hi)
        if mask.sum() > 1:
            vals = all_B[mask]
            median.append(np.median(vals))
            p16.append(np.percentile(vals, 16))
            p84.append(np.percentile(vals, 84))
        else:
            median.append(np.nan)
            p16.append(np.nan)
            p84.append(np.nan)

    median = np.array(median)
    p16, p84 = np.array(p16), np.array(p84)

    # ── Plot ─────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))

    # Individual tracks (faint)
    for gal in all_galaxies:
        if not gal["infall"]:
            continue
        mpb  = gal["mpb"]
        valid = [r for r in mpb
                 if "dt_infall" in r and np.isfinite(r["B_physical"]) and r["B_physical"] > 0]
        if not valid:
            continue
        dt_g = [r["dt_infall"]  for r in valid]
        B_g  = [r["B_physical"] for r in valid]
        order = np.argsort(dt_g)
        ax.plot(np.array(dt_g)[order], np.array(B_g)[order],
                color="steelblue", alpha=0.15, lw=1)

    # Median + scatter band
    valid_bins = np.isfinite(median)
    ax.fill_between(bin_c[valid_bins], p16[valid_bins], p84[valid_bins],
                    alpha=0.35, color="tomato", label="16th–84th percentile")
    ax.plot(bin_c[valid_bins], median[valid_bins],
            color="tomato", lw=2.5, label="Median")

    # Infall marker
    ax.axvline(0, color="black", lw=2, ls="--", label="Infall ($\\Delta t = 0$)")

    ax.set_yscale("log")
    ax.set_xlabel(r"$\Delta t = t_{\rm lookback} - t_{\rm infall}$ [Gyr]", fontsize=13)
    ax.set_ylabel(r"$B_{\rm phys}$ [G]", fontsize=13)
    ax.set_title("Magnetic Field vs Time Since Infall — TNG100 Satellites", fontsize=14)

    # Annotate pre/post infall
    ax.text(2, ax.get_ylim()[0] * 2, "← before infall", fontsize=10, color="gray")
    ax.text(-3.5, ax.get_ylim()[0] * 2, "after infall →", fontsize=10, color="gray")

    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("results/figures/B_vs_dt_infall.png", dpi=150)
    plt.show()


plot_B_vs_dt_infall(all_galaxies)

---
## 10. (Reach Goal) Fit a Growth Model to Post-Infall B-field

Fit B(t) = B₀ · exp(t / τ) to the post-infall median track, where τ is the **amplification timescale**.

In [ ]:
from scipy.optimize import curve_fit

def exponential_growth(t, B0, tau):
    """B0 * exp(-t / tau)  — note: t here is dt_infall (positive = before infall)."""
    return B0 * np.exp(-t / tau)


def fit_amplification_timescale(all_galaxies, dt_range=(-4, 0), bin_width=0.5):
    """
    Fit an exponential to the post-infall (dt < 0) segment of the median B-field.
    """
    # Recompute the binned median (post-infall only)
    all_dt, all_B = [], []
    for gal in all_galaxies:
        if not gal["infall"]:
            continue
        for r in gal["mpb"]:
            if "dt_infall" in r and r["dt_infall"] < 0 and np.isfinite(r["B_physical"]) and r["B_physical"] > 0:
                all_dt.append(r["dt_infall"])
                all_B.append(r["B_physical"])

    all_dt = np.array(all_dt)
    all_B  = np.array(all_B)

    bins  = np.arange(dt_range[0], dt_range[1] + bin_width, bin_width)
    bin_c = 0.5 * (bins[:-1] + bins[1:])
    median = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (all_dt >= lo) & (all_dt < hi)
        median.append(np.median(all_B[mask]) if mask.sum() > 1 else np.nan)

    median  = np.array(median)
    valid   = np.isfinite(median)
    x, y    = bin_c[valid], median[valid]

    if len(x) < 3:
        print("Not enough bins to fit — get more galaxies!")
        return None

    try:
        popt, pcov = curve_fit(exponential_growth, x, y,
                               p0=[y[-1], 2.0], maxfev=5000)
        B0_fit, tau_fit = popt
        perr = np.sqrt(np.diag(pcov))
        print(f"Best-fit amplification timescale: τ = {tau_fit:.2f} ± {perr[1]:.2f} Gyr")
        print(f"B at infall (B₀):                B₀ = {B0_fit:.3e} G")

        # Plot
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.scatter(x, y, color="tomato", zorder=3, label="Binned median")
        t_fine  = np.linspace(x.min(), 0, 300)
        ax.plot(t_fine, exponential_growth(t_fine, *popt),
                "k--", lw=2, label=rf"Fit: $\tau = {tau_fit:.2f}$ Gyr")
        ax.axvline(0, color="gray", ls=":", label="Infall")
        ax.set_yscale("log")
        ax.set_xlabel(r"$\Delta t$ [Gyr]  (negative = post-infall)", fontsize=12)
        ax.set_ylabel(r"$B_{\rm phys}$ [G]", fontsize=12)
        ax.set_title("Post-Infall Magnetic Field Amplification Fit", fontsize=13)
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig("results/figures/B_amplification_fit.png", dpi=150)
        plt.show()

    except RuntimeError as e:
        print(f"Fit failed: {e}")


fit_amplification_timescale(all_galaxies)

---
## Notes & Next Steps

### Unit clarification
Double-check the physical B-field conversion against the [TNG field specifications](https://www.tng-project.org/data/docs/specifications/#sec2a). The field `SubhaloMagneticField` definition can vary between TNG runs.

### Increasing your sample
Change `limit=10` in `query_satellites()` to a larger number (e.g. 100–500) once you're happy with the code. Expect API calls to take ~1–2 seconds each.

### Caching
Consider saving processed MPB data to disk (e.g. via `numpy.save` or `pickle`) so you don't re-query the API every run.

```python
import pickle
with open('data/processed/all_galaxies.pkl', 'wb') as f:
    pickle.dump(all_galaxies, f)
# Reload:
with open('data/processed/all_galaxies.pkl', 'rb') as f:
    all_galaxies = pickle.load(f)
```

### Stratified analysis (reach goal)
Split the sample by stellar mass bins and compare amplification timescales across mass bins.